# Improved PodcastFillers classifier — Kaggle GPU notebook

This notebook trains a stronger six-class filler/audio-event classifier from the uploaded PodcastFillers dataset. It is deliberately **training-only** (no existing checkpoints are required): the model uses a pretrained wav2vec2 acoustic encoder, unfreezes its upper transformer layers, and adds a temporal CNN plus attentive pooling head.

Compared with the original frozen-embedding linear probe, this version adds: waveform augmentation, partial end-to-end fine-tuning, class-balanced focal loss, mixed precision, early stopping, calibration, and class-wise diagnostics. The inherited train/validation/test split is preserved to avoid contaminating the final test set.

In [ ]:
# 1. Configuration and reproducibility
import os, sys, json, csv, random, warnings, math, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE = 16_000
CLIP_SAMPLES = SAMPLE_RATE
CLASSES = ["Words", "Uh", "Um", "Breath", "Laughter", "Music"]
CLASS_TO_IDX = {x:i for i,x in enumerate(CLASSES)}
MODEL_NAME = "facebook/wav2vec2-base"
OUTPUT_DIR = Path("/kaggle/working/filler_detection_improved") if Path("/kaggle/working").exists() else Path("filler_detection_improved")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EPOCHS = 12
BATCH_SIZE = 32 if DEVICE.type == "cuda" else 4
GRAD_ACCUM = 2 if DEVICE.type == "cuda" else 1
LR_ENCODER = 1e-5
LR_HEAD = 3e-4
WEIGHT_DECAY = 1e-2
PATIENCE = 3
NUM_WORKERS = min(4, os.cpu_count() or 1)
print({"device":str(DEVICE), "model":MODEL_NAME, "batch_size":BATCH_SIZE, "output":str(OUTPUT_DIR)})

## 2. Locate the uploaded dataset
Kaggle may mount a dataset under a generated directory name, so this searches recursively for the annotation/audio contract rather than assuming one absolute path.

In [ ]:
KAGGLE_DATA_ROOT = Path("/kaggle/input/datasets/whitedevilanand/podcastfillers/PodcastFillers")

def find_dataset_root():
    # Primary location for the attached Kaggle dataset.
    if KAGGLE_DATA_ROOT.is_dir():
        return KAGGLE_DATA_ROOT
    candidates = [Path("/kaggle/input"), Path("/kaggle/input/datasets"), Path("PodcastFillers"), Path(".")]
    checked = set()
    for base in candidates:
        if not base.exists(): continue
        for ann in base.rglob("episode_annotations"):
            root = ann.parent.parent
            if all((ann / s).is_dir() for s in ("train","validation","test")) and (root / "audio" / "clip_wav").exists():
                return root
            checked.add(str(ann))
    raise FileNotFoundError("Could not find PodcastFillers/metadata/episode_annotations and audio/clip_wav under /kaggle/input or the current directory.")
DATA_ROOT = find_dataset_root()
ANN_ROOT = DATA_ROOT / "metadata" / "episode_annotations"
CLIP_ROOT = DATA_ROOT / "audio" / "clip_wav"
print("DATA_ROOT:", DATA_ROOT)
print("Annotation splits:", [p.name for p in ANN_ROOT.iterdir() if p.is_dir()])

## 3. Read annotations and validate the data
Only the six consolidated labels used by the baseline are retained. Missing/corrupt clips are reported and excluded instead of silently producing incorrect labels.

In [ ]:
def load_split(split):
    rows=[]; missing=[]; unknown=0
    for csv_path in sorted((ANN_ROOT/split).glob("*.csv")):
        with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
            for row in csv.DictReader(f):
                label=row.get("label_consolidated_vocab", "")
                if label not in CLASS_TO_IDX:
                    unknown += 1; continue
                wav=CLIP_ROOT/split/row["clip_name"]
                if not wav.is_file(): missing.append(str(wav)); continue
                rows.append({"wav":str(wav), "label":CLASS_TO_IDX[label], "class":label, "episode":row.get("podcast_filename", csv_path.stem), "confidence":float(row.get("confidence") or 1.0), "duration":float(row.get("duration") or 0.0)})
    df=pd.DataFrame(rows)
    if df.empty: raise RuntimeError(f"No usable records in {split}")
    print(f"{split:10s}: {len(df):6d} clips | missing={len(missing)} | excluded_labels={unknown}")
    print(df["class"].value_counts().reindex(CLASSES, fill_value=0).to_dict())
    return df
train_df=load_split("train"); val_df=load_split("validation"); test_df=load_split("test")
print("Episode overlap train/val:", len(set(train_df.episode)&set(val_df.episode)), "train/test:", len(set(train_df.episode)&set(test_df.episode)), "val/test:", len(set(val_df.episode)&set(test_df.episode)))

In [ ]:
# Distribution and quality overview
import matplotlib.pyplot as plt
fig, ax=plt.subplots(1,2,figsize=(14,4))
for df,name in [(train_df,"Train"),(val_df,"Validation"),(test_df,"Test")]:
    df["class"].value_counts().reindex(CLASSES,fill_value=0).plot(kind="bar",ax=ax[0],alpha=.65,label=name)
ax[0].set_ylabel("clips"); ax[0].set_title("Class distribution"); ax[0].legend()
ax[1].bar(CLASSES, train_df["class"].value_counts().reindex(CLASSES,fill_value=0).values)
ax[1].set_title("Training distribution"); ax[1].tick_params(axis="x",rotation=35)
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"class_distribution.png",dpi=160); plt.show()

## 4. Dataset, augmentation, and feature extractor
Clips are resampled/mono-normalized and padded or cropped to one second. Augmentation is applied only to training examples: random gain, additive background-like noise, and time masking. The validation/test waveforms remain untouched.

In [ ]:
import soundfile as sf
import librosa
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
extractor=Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)

def read_audio(path):
    audio,sr=sf.read(path, dtype="float32", always_2d=False)
    if audio.ndim>1: audio=audio.mean(axis=1)
    if sr != SAMPLE_RATE: audio=librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
    audio=np.asarray(audio,dtype=np.float32)
    audio=np.nan_to_num(audio)
    if len(audio)<CLIP_SAMPLES: audio=np.pad(audio,(0,CLIP_SAMPLES-len(audio)))
    elif len(audio)>CLIP_SAMPLES:
        start=np.random.randint(0,len(audio)-CLIP_SAMPLES+1)
        audio=audio[start:start+CLIP_SAMPLES]
    peak=np.max(np.abs(audio))+1e-8
    audio=np.clip(audio/(peak if peak>1 else 1),-1,1)
    return audio

def augment(audio):
    audio=audio.copy()
    audio *= np.random.uniform(0.75,1.15)
    if np.random.rand()<0.35:
        audio += np.random.randn(len(audio)).astype(np.float32)*np.random.uniform(0.001,0.008)
    if np.random.rand()<0.25:
        a=np.random.randint(0,len(audio)//3); b=min(len(audio),a+np.random.randint(200,1800)); audio[a:b]*=np.random.uniform(0.0,0.25)
    return np.clip(audio,-1,1).astype(np.float32)

class FillerDataset(Dataset):
    def __init__(self,df,training=False): self.df=df.reset_index(drop=True); self.training=training
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; audio=read_audio(r.wav)
        if self.training: audio=augment(audio)
        return {"audio":torch.from_numpy(audio), "label":torch.tensor(int(r.label)), "path":r.wav}

def collate(batch):
    return {"audio":torch.stack([x["audio"] for x in batch]), "label":torch.stack([x["label"] for x in batch]), "path":[x["path"] for x in batch]}
train_ds=FillerDataset(train_df,True); val_ds=FillerDataset(val_df); test_ds=FillerDataset(test_df)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True,collate_fn=collate,persistent_workers=NUM_WORKERS>0)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE*2,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,collate_fn=collate,persistent_workers=NUM_WORKERS>0)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE*2,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,collate_fn=collate,persistent_workers=NUM_WORKERS>0)

## 5. Stronger model
The wav2vec2 feature extractor is frozen for stability, while the upper transformer blocks learn task-specific acoustics. A depthwise temporal CNN captures local phonetic/noise patterns; attentive statistics pooling retains both where an event occurs and its variability. The classifier consumes mean and standard-deviation pooled representations.

In [ ]:
class AttentiveStatsPool(nn.Module):
    def __init__(self,dim):
        super().__init__(); self.attn=nn.Sequential(nn.Linear(dim,dim//2),nn.Tanh(),nn.Linear(dim//2,1))
    def forward(self,x,mask=None):
        a=self.attn(x).squeeze(-1)
        if mask is not None: a=a.masked_fill(~mask,-1e4)
        a=torch.softmax(a,dim=-1).unsqueeze(-1)
        mean=(a*x).sum(1); var=(a*(x-mean.unsqueeze(1)).pow(2)).sum(1)
        return torch.cat([mean,torch.sqrt(var+1e-5)],dim=-1)

class Wav2VecTemporalClassifier(nn.Module):
    def __init__(self,n_classes=6):
        super().__init__(); self.encoder=Wav2Vec2Model.from_pretrained(MODEL_NAME)
        self.encoder.feature_extractor._freeze_parameters()
        for layer in self.encoder.encoder.layers[:-4]:
            for p in layer.parameters(): p.requires_grad=False
        d=self.encoder.config.hidden_size
        self.temporal=nn.Sequential(nn.Conv1d(d,d,5,padding=2,groups=d),nn.GELU(),nn.BatchNorm1d(d),nn.Conv1d(d,256,1),nn.GELU(),nn.Dropout(.15))
        self.pool=AttentiveStatsPool(256)
        self.head=nn.Sequential(nn.LayerNorm(512),nn.Linear(512,256),nn.GELU(),nn.Dropout(.30),nn.Linear(256,n_classes))
    def forward(self,audio):
        z=self.encoder(audio).last_hidden_state
        z=self.temporal(z.transpose(1,2)).transpose(1,2)
        return self.head(self.pool(z))
model=Wav2VecTemporalClassifier(len(CLASSES)).to(DEVICE)
print(f"trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Balanced focal training
The loss mildly prioritizes underrepresented classes without discarding the naturally occurring distribution. Gradient clipping, AMP, a warmup/cosine schedule, and early stopping make the large GPU run stable.

In [ ]:
counts=np.bincount(train_df.label.values,minlength=len(CLASSES)).astype(np.float32)
weights=(counts.sum()/(len(CLASSES)*counts)); weights=weights/weights.mean()
class_weights=torch.tensor(weights,dtype=torch.float32,device=DEVICE)
class BalancedFocalLoss(nn.Module):
    def __init__(self,weights,gamma=1.5): super().__init__(); self.weights=weights; self.gamma=gamma
    def forward(self,logits,target):
        ce=F.cross_entropy(logits,target,weight=self.weights,reduction="none"); pt=torch.exp(-ce); return (((1-pt)**self.gamma)*ce).mean()
criterion=BalancedFocalLoss(class_weights)
enc_params=[p for p in model.encoder.parameters() if p.requires_grad]; head_params=[p for n,p in model.named_parameters() if not n.startswith("encoder.") and p.requires_grad]
optimizer=torch.optim.AdamW([{"params":enc_params,"lr":LR_ENCODER},{"params":head_params,"lr":LR_HEAD}],weight_decay=WEIGHT_DECAY)
steps=math.ceil(len(train_loader)/GRAD_ACCUM)*EPOCHS
scheduler=torch.optim.lr_scheduler.OneCycleLR(optimizer,max_lr=[LR_ENCODER,LR_HEAD],total_steps=steps,pct_start=.1,anneal_strategy="cos")
scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda")
config={"model_name":MODEL_NAME,"classes":CLASSES,"sample_rate":SAMPLE_RATE,"epochs":EPOCHS,"batch_size":BATCH_SIZE,"grad_accum":GRAD_ACCUM,"class_weights":weights.tolist()}
(OUTPUT_DIR/"config.json").write_text(json.dumps(config,indent=2))

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
def run_epoch(loader,training):
    model.train(training); losses=[]; ys=[]; ps=[]
    if not training: torch.set_grad_enabled(False)
    for step,b in enumerate(loader):
        x=b["audio"].to(DEVICE,non_blocking=True); y=b["label"].to(DEVICE,non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type=="cuda"):
            logits=model(x); loss=criterion(logits,y)
        if training:
            scaler.scale(loss/GRAD_ACCUM).backward()
            if (step+1)%GRAD_ACCUM==0 or step+1==len(loader):
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()
        losses.append(loss.item()); ys.extend(y.detach().cpu().numpy()); ps.extend(logits.detach().argmax(1).cpu().numpy())
    if not training: torch.set_grad_enabled(True)
    return float(np.mean(losses)),accuracy_score(ys,ps),f1_score(ys,ps,average="macro")

history=[]; best=-1; bad=0
for epoch in range(1,EPOCHS+1):
    tr=run_epoch(train_loader,True); va=run_epoch(val_loader,False)
    rec={"epoch":epoch,"train_loss":tr[0],"train_acc":tr[1],"train_macro_f1":tr[2],"val_loss":va[0],"val_acc":va[1],"val_macro_f1":va[2]}; history.append(rec); print(rec)
    if va[2]>best:
        best=va[2]; bad=0; torch.save({"model_state":model.state_dict(),"config":config,"epoch":epoch,"val_macro_f1":best},OUTPUT_DIR/"filler_wav2vec_temporal_best.pt"); print("  saved best checkpoint")
    else: bad+=1
    if bad>=PATIENCE: print("Early stopping"); break
(OUTPUT_DIR/"training_history.json").write_text(json.dumps(history,indent=2))
model.load_state_dict(torch.load(OUTPUT_DIR/"filler_wav2vec_temporal_best.pt",map_location=DEVICE)["model_state"])

In [ ]:
# Training curves
h=pd.DataFrame(history); fig,ax=plt.subplots(1,3,figsize=(16,4))
ax[0].plot(h.epoch,h.train_loss,label="train"); ax[0].plot(h.epoch,h.val_loss,label="validation"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(h.epoch,h.train_acc,label="train"); ax[1].plot(h.epoch,h.val_acc,label="validation"); ax[1].set_title("Accuracy"); ax[1].legend()
ax[2].plot(h.epoch,h.train_macro_f1,label="train"); ax[2].plot(h.epoch,h.val_macro_f1,label="validation"); ax[2].set_title("Macro F1"); ax[2].legend()
for a in ax:a.set_xlabel("epoch");a.grid(alpha=.2)
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"training_curves.png",dpi=180); plt.show()

## 7. Comprehensive held-out test evaluation
The final checkpoint is selected only by validation macro-F1. The test set is evaluated once with per-class precision/recall/F1, macro and weighted aggregates, normalized confusion matrices, one-vs-rest ROC/PR curves, and confidence calibration.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.calibration import calibration_curve
def predict(loader):
    model.eval(); ys=[]; probs=[]; paths=[]
    with torch.no_grad():
        for b in loader:
            with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=DEVICE.type=="cuda"):
                q=torch.softmax(model(b["audio"].to(DEVICE)),dim=1)
            ys.extend(b["label"].numpy()); probs.append(q.float().cpu().numpy()); paths.extend(b["path"])
    return np.asarray(ys),np.concatenate(probs),paths
y_true,y_prob,test_paths=predict(test_loader); y_pred=y_prob.argmax(1)
report=classification_report(y_true,y_pred,target_names=CLASSES,output_dict=True,zero_division=0)
summary={"accuracy":float(accuracy_score(y_true,y_pred)),"macro_f1":float(f1_score(y_true,y_pred,average="macro")),"weighted_f1":float(f1_score(y_true,y_pred,average="weighted")),"per_class":report}
print(pd.DataFrame(report).T.round(4)); print(json.dumps({k:summary[k] for k in ("accuracy","macro_f1","weighted_f1")},indent=2))
(OUTPUT_DIR/"test_metrics.json").write_text(json.dumps(summary,indent=2))
pd.DataFrame({"path":test_paths,"true":y_true,"pred":y_pred,"confidence":y_prob.max(1),**{f"p_{c}":y_prob[:,i] for i,c in enumerate(CLASSES)}}).to_csv(OUTPUT_DIR/"test_predictions.csv",index=False)

In [ ]:
# Confusion matrix and ROC/PR diagnostics
cm=confusion_matrix(y_true,y_pred,labels=np.arange(len(CLASSES))); cmn=cm/cm.sum(1,keepdims=True)
fig,ax=plt.subplots(1,3,figsize=(20,5))
for a,m,title,fmt in [(ax[0],cm,"Counts", "d"),(ax[1],cmn,"Row-normalized", ".2f")]:
    im=a.imshow(m,cmap="Blues",vmin=0); a.set_xticks(range(6),CLASSES,rotation=35,ha="right"); a.set_yticks(range(6),CLASSES); a.set_title(title)
    for i in range(6):
        for j in range(6): a.text(j,i,format(m[i,j],fmt),ha="center",va="center",color="white" if m[i,j]>m.max()*.55 else "black")
    a.set_xlabel("predicted"); a.set_ylabel("true")
for i,c in enumerate(CLASSES):
    fpr,tpr,_=roc_curve((y_true==i),y_prob[:,i]); ax[2].plot(fpr,tpr,label=f"{c} AUC={auc(fpr,tpr):.3f}")
ax[2].plot([0,1],[0,1],"k--",alpha=.4); ax[2].set_title("One-vs-rest ROC"); ax[2].legend(fontsize=8); ax[2].set_xlabel("false positive rate"); ax[2].set_ylabel("true positive rate")
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"confusion_and_roc.png",dpi=180); plt.show()
fig,ax=plt.subplots(figsize=(8,6))
for i,c in enumerate(CLASSES):
    p,r,_=precision_recall_curve(y_true==i,y_prob[:,i]); ax.plot(r,p,label=f"{c} AP={average_precision_score(y_true==i,y_prob[:,i]):.3f}")
ax.set(xlabel="recall",ylabel="precision",title="One-vs-rest precision-recall"); ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"precision_recall.png",dpi=180); plt.show()

In [ ]:
# Confidence calibration and qualitative errors
conf=y_prob.max(1); correct=(y_pred==y_true); bins=np.linspace(0,1,11); centers=[]; accs=[]; counts_bin=[]
for lo,hi in zip(bins[:-1],bins[1:]):
    sel=(conf>=lo)&(conf<hi if hi<1 else conf<=hi)
    if sel.any(): centers.append(conf[sel].mean()); accs.append(correct[sel].mean()); counts_bin.append(sel.sum())
fig,ax=plt.subplots(1,2,figsize=(13,5)); ax[0].plot(centers,accs,"o-"); ax[0].plot([0,1],[0,1],"k--"); ax[0].set(xlabel="confidence",ylabel="accuracy",title="Reliability diagram")
ax[1].hist(conf[correct],bins=20,alpha=.7,label="correct"); ax[1].hist(conf[~correct],bins=20,alpha=.7,label="incorrect"); ax[1].set_title("Prediction confidence"); ax[1].legend(); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"calibration.png",dpi=180); plt.show()
errors=np.where(~correct)[0]; errors=errors[np.argsort(-conf[errors])][:30]
err=pd.DataFrame({"path":[test_paths[i] for i in errors],"true":[CLASSES[y_true[i]] for i in errors],"pred":[CLASSES[y_pred[i]] for i in errors],"confidence":conf[errors]}); display(err); err.to_csv(OUTPUT_DIR/"high_confidence_errors.csv",index=False)

## 8. Export summary
The output directory contains the best checkpoint, exact label/configuration metadata, training history, predictions, metrics, plots, and a high-confidence error table. The checkpoint can be loaded later without retraining.

In [ ]:
print("Artifacts written to:", OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.iterdir()): print(f"{p.name:35s} {p.stat().st_size/1024:.1f} KB")